<a href="https://colab.research.google.com/github/hasnanasa/AI-projects/blob/main/translationlanguage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this project I worked hard totranslate a texte that i write into a vocal voice
either i choose a language : arabic and my text was in french , i am gonna hear my vthe text with my voice in arabic

In [53]:
!rm -rf /content/venv
print("✅ Cleaned")

✅ Cleaned


In [54]:
!apt-get update -qq
!apt-get install -y python3.10 python3.10-venv -qq

!python3.10 -m venv /content/venv
!/content/venv/bin/pip install --upgrade pip

# Install PyTorch 2.4 (NOT 2.6+)
!/content/venv/bin/pip install torch==2.4.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu118

# Install transformers compatible version
!/content/venv/bin/pip install transformers==4.31.0

# Install TTS
!/content/venv/bin/pip install TTS

# Install other packages
!/content/venv/bin/pip install gradio deep-translator langdetect

print("✅ Setup complete!")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
  Using cached pip-26.1.2-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 22.0.2
    Uninstalling pip-22.0.2:
      Successfully uninstalled pip-22.0.2
Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.7/857.7 MB 29.7 MB/s  0:00:14
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 88.3 MB/s  0:00:00
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.4.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.4.0-py3-none-any.whl.metadata (10 kB)
     

In [57]:
!/content/venv/bin/pip uninstall transformers -y
!/content/venv/bin/pip install transformers==4.31.0

Found existing installation: transformers 5.10.2
Uninstalling transformers-5.10.2:
  Successfully uninstalled transformers-5.10.2
  Using cached transformers-4.31.0-py3-none-any.whl.metadata (116 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached tokenizers-0.13.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
Using cached transformers-4.31.0-py3-none-any.whl (7.4 MB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
Using cached tokenizers-0.13.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (7.8 MB)
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.18.0
    Uninstalling huggingface_hub-1.18.0:
      Successfully uninstalled huggingface_hub-1.18.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [58]:
%%writefile /content/app.py
import os
os.environ['COQUI_TOS_AGREED'] = '1'
os.environ['MPLBACKEND'] = 'Agg'

# Fix for PyTorch weights_only issue
import torch
torch.serialization.add_safe_globals(['TTS.tts.configs.xtts_config.XttsConfig'])

from TTS.api import TTS
import gradio as gr
from deep_translator import GoogleTranslator
import re
import glob

print("="*50)
print("Loading Voice Model...")
print("="*50)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

# Load model
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
print("✅ Model loaded!")

# Find your voice
voice_files = glob.glob('/content/drive/MyDrive/my_voice_dataset/**/*.wav', recursive=True)
if voice_files:
    YOUR_VOICE = voice_files[0]
    print(f"✅ Voice: {YOUR_VOICE}")
else:
    YOUR_VOICE = None
    print("❌ No voice found!")

def detect_language(text):
    if re.search(r'[\u0600-\u06FF]', text):
        return "Arabic"
    elif re.search(r'[éèêëàâçôûîï]', text.lower()):
        return "French"
    return "English"

def translate_text(text, source, target):
    codes = {"Arabic": "ar", "English": "en", "French": "fr"}
    try:
        translator = GoogleTranslator(source=codes[source], target=codes[target])
        return translator.translate(text)
    except:
        return text

def process(text, target):
    if YOUR_VOICE is None:
        return None, "No voice!"

    source = detect_language(text)
    if source == target:
        translated = text
        info = f"Same: {source}"
    else:
        translated = translate_text(text, source, target)
        info = f"{source} -> {target}"

    codes = {"Arabic": "ar", "English": "en", "French": "fr"}
    out = "/content/output.wav"

    tts.tts_to_file(
        text=translated,
        speaker_wav=YOUR_VOICE,
        language=codes[target],
        file_path=out
    )

    return out, f"{info}\n\n'{text}' -> '{translated}'"

gr.Interface(
    fn=process,
    inputs=[gr.Textbox(label="Type", lines=2), gr.Dropdown(["English","Arabic","French"], label="Output")],
    outputs=[gr.Audio(label="Your voice", autoplay=True), gr.Textbox(label="Info", lines=4)],
    title="Voice Translator"
).launch(share=True)

Overwriting /content/app.py


In [ ]:

!source /content/venv/bin/activate && python /content/app.py

Loading Voice Model...
Device: cuda
PyTorch: 2.4.0+cu118
 > tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.
 > Using model: xtts
/content/venv/lib/python3.10/site-packages/TTS/utils/io.py:54: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. 

In [ ]:
!/content/venv/bin/pip uninstall TTS -y
!/content/venv/bin/pip install TTS==0.20.0
!/content/venv/bin/pip install transformers==4.31.0

In [46]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
